# 02 · Solapamiento de pacientes al partir por admisión — Muerte materna (M1) → morbilidad materna severa (M2), MIMIC-IV

| Campo | Contenido |
|---|---|
| Secuencia de notebooks | 01: E1 · E2 · E3 · E4 · P1 → **02: E5 · P2** → 03: E6 · E7 · P3 · R1 · E8 → 04: E8 → 05: E9 (E7) → 06: E9 (E3) → 07: E9 · C1-C3 · R2 · R4 → 08: E9 · C4 → 09: E9 · C4 → 10: P4 · E10 · E11 |
| Qué hace | Cuenta cuántas pacientes (y pacientes con evento) quedan a ambos lados de la frontera de evaluación cuando la primera versión del análisis partió los datos por admisión; es la evidencia para agrupar todas las particiones por paciente (notebook 03). No cambia ningún modelo, etiqueta ni conjunto de datos |
| Etapas y puertas | E1 · E4 (réplica de la primera versión) · E5 unidad de análisis y solapamiento · P2 (ninguna paciente a ambos lados) |
| Entradas | Base local PostgreSQL de MIMIC-IV (tablas de `sql/`): `cohort_maternal_dataset`, `cohort_smm_dataset` (solo `subject_id`, `hadm_id`, desenlace) |
| Salidas | Ninguna (no escribe archivos) |
| Uso de los datos | Solo se muestran y guardan resultados agregados; nada a nivel de paciente. No agregar celdas que muestren o exporten filas |
| Tiempo | Unos minutos (1.000 + 200 particiones simuladas) |
| Siguiente | `03_grouped_validation_smm_es.ipynb` |
| Versión en inglés | `02_patient_overlap.ipynb` (mismo código) |

## E1 · Gobierno y configuración — librerías y conexión

In [1]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- language and presentation
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
# Nothing below changes a computation: it only decides how tables and messages are shown. The English aggregate
# reports data/outputs_*.txt are written exactly as before (the article cites them); the Spanish notebook also
# writes a translated copy, data/outputs_*_es.txt.
LANG = "es"
import builtins, contextlib, importlib, io, html as _html
import numpy as np
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)
def _tr(s, lang=None):
    """Translate one line keeping its indentation and trailing spaces."""
    lang = lang or LANG
    if lang != "es" or not isinstance(s, str):
        return s
    out = []
    for ln in s.split("\n"):
        core = ln.strip()
        if not core:
            out.append(ln); continue
        i = ln.index(core)
        out.append(ln[:i] + translate(core, "es") + ln[i + len(core):])
    return "\n".join(out)

_REC = None          # while emit() runs: the blocks of the English report, [("text", line) | ("table", frame, index)]

def print(*args, sep=" ", end="\n", file=None, flush=False):
    """Inside emit() the English text goes to the report unchanged; outside it, messages follow LANG."""
    if _REC is not None and file is None:
        _REC.append(("text", sep.join(str(a) for a in args)))
        return builtins.print(*args, sep=sep, end=end, flush=flush)
    if LANG == "es":
        args = [_tr(a, "es") for a in args]
    return builtins.print(*args, sep=sep, end=end, file=file, flush=flush)

def _cell_fmt(v):
    # Display format of a cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer() and abs(v) >= 1000:
            return f"{int(v):,}"
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _html_table(df, index):
    df = df.astype(object)
    df = df.map(_cell_fmt) if hasattr(df, "map") else df.applymap(_cell_fmt)
    return f"<div style='max-height:420px; overflow:auto'>{df.to_html(index=index, na_rep='', border=0)}</div>"

def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

def table(df, index=False, title=None):
    """Show a table. Inside emit() it also prints the English text of the report (df.to_string)."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
        index = True
    if _REC is not None:
        _REC.append(("table", df, index))
        return builtins.print(df.to_string(index=index))
    if not _in_notebook():
        return builtins.print(translate(df, LANG).to_string(index=index))
    from IPython.display import HTML, display as _display
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>" if title else ""
    _display(HTML(head + _html_table(translate(df, LANG), index)))

def _blocks_html(blocks):
    """Report blocks as HTML: a text line right before a table is its title; rules of '=' are dropped."""
    out, pending = [], []
    def flush_text():
        for ln in pending:
            if ln.strip():
                out.append(f"<div style='white-space:pre-wrap; font-family:monospace; margin:2px 0'>{_html.escape(_tr(ln.rstrip()))}</div>")
        pending.clear()
    for b in blocks:
        if b[0] == "text":
            pending.extend(l for l in b[1].split("\n") if not set(l.strip()) <= {"="} or not l.strip())
        else:
            title = None
            while pending and not pending[-1].strip():
                pending.pop()
            if pending:
                title = pending.pop().strip()
            flush_text()
            if title:
                out.append(f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>")
            out.append(_html_table(translate(b[1], LANG), b[2]))
    flush_text()
    return "\n".join(out)

def _blocks_text_es(blocks):
    """Spanish copy of a report (data/outputs_*_es.txt)."""
    lines = []
    for b in blocks:
        if b[0] == "text":
            lines.extend(_tr(l, "es") for l in b[1].split("\n"))
        else:
            lines.extend(translate(b[1], "es").to_string(index=b[2]).split("\n"))
    return "\n".join(lines) + "\n"

def _emit_to(path, fn, show=True):
    """Run fn, write its English text to `path` exactly as printed and, from the Spanish notebook, a translated copy
    (*_es.txt); show the report as tables."""
    global _REC
    buf = io.StringIO()
    _REC = []
    try:
        with contextlib.redirect_stdout(buf):
            fn()
    finally:
        blocks, _REC = _REC, None
    text = buf.getvalue()
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    if show:
        show_report(blocks)
    if LANG == "es":
        with open(path.replace(".txt", "_es.txt"), "w", encoding="utf-8") as fh:
            fh.write(_blocks_text_es(blocks))
    return text

def show_report(blocks):
    if _in_notebook():
        from IPython.display import HTML, display as _display
        _display(HTML(_blocks_html(blocks)))
    else:
        for b in blocks:
            builtins.print(_tr(b[1]) if b[0] == "text" else translate(b[1], LANG).to_string(index=b[2]))

def notebook_summary(rows, title="Notebook summary"):
    """Summary of the notebook (as in the neonatal repository): one value per row; a value that cannot be computed
    (for example, an analysis that was not run in this session) is shown as n/a."""
    out = {}
    for label, fn in rows:
        try:
            v = fn() if callable(fn) else fn
        except Exception:
            v = "n/a"
        out[label] = v
    table(pd.Series(out, name="value").to_frame(), index=True, title=title)

def _m(res, model, met, d=3):
    """Point value of a metric with its 95% bootstrap interval, from a result with "point" and "boot"."""
    v = f"{res['point'][(model, met)]:.{d}f}"
    b = res.get("boot") or {}
    a = b.get((model, met)) if isinstance(b, dict) else None
    if a is not None and len(a) >= 20:
        lo, hi = np.nanpercentile(a, [2.5, 97.5])
        v += f" ({lo:.{d}f}-{hi:.{d}f})"
    return v

def _cv_auroc(res, model):
    """AUROC averaged over the repeats of a cross-validation result that only keeps out-of-fold predictions."""
    from sklearn.metrics import roc_auc_score
    y = res["y"]
    return float(np.mean([roc_auc_score(y, res["preds"][model][r]) for r in range(len(res["preds"][model]))]))


## E4 · Auditoría de conteos — identificadores y etiquetas de la primera versión

Solo se leen tres columnas de las dos tablas de la primera versión (`sql/02_cohort_mortality/maternal_cohort_dataset.sql` y `sql/03_cohort_smm/smm_dataset.sql`); leer menos columnas no cambia el orden de las filas de una lectura secuencial. Los conteos de positivos esperados (41 y 475) confirman que son las etiquetas de la primera versión.

In [2]:
SPECS = {
    "A": dict(name="Model A - mortality (baseline label)",
              query="SELECT subject_id, hadm_id, outcome FROM mimiciv.cohort_maternal_dataset;",
              expected_pos=41, expected_train=(20588, 33), expected_test=(5148, 8)),
    "B": dict(name="Model B - SMM (baseline label)",
              query="SELECT subject_id, hadm_id, outcome_smm AS outcome FROM mimiciv.cohort_smm_dataset;",
              expected_pos=475, expected_train=(20588, 380), expected_test=(5148, 95)),
}

raw, load_rows = {}, []
for key, sp in SPECS.items():
    d = pd.read_sql(sp["query"], engine)
    d["outcome"] = d["outcome"].astype(int)
    raw[key] = d
    ok = "OK" if int(d["outcome"].sum()) == sp["expected_pos"] else "CHECK: differs from expected count"
    load_rows.append({"model": sp["name"], "admissions": len(d), "patients": d["subject_id"].nunique(),
                      "positive admissions": int(d["outcome"].sum()), "expected positive admissions": sp["expected_pos"],
                      "check": ok})
table(pd.DataFrame(load_rows), title="Tables of the first version, loaded")

modelo,admisiones,pacientes,admisiones positivas,admisiones positivas esperadas,verificación
Modelo A - mortalidad (etiqueta de la primera versión),"25,736","15,084",41,41,OK
Modelo B - SMM (etiqueta de la primera versión),"25,736","15,084",475,475,OK


## E4 · Auditoría de conteos — réplica del orden de filas de la primera versión

La primera versión ordenó por `subject_id` (orden por defecto, inestable en los empates) antes de partir; aquí se aplica la misma instrucción. Si coinciden los tamaños y positivos de entrenamiento y prueba, la réplica es consistente (no prueba que las filas sean idénticas una a una; la sección E5 da el rango independiente del orden).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| `coincide` es Sí en los dos modelos | Sigue: se puede medir el solapamiento de la primera versión | Sí en A y B |
| `coincide` es No | Cambió el orden de las filas: usar solo la simulación (independiente del orden) | — |

In [3]:
def ordered(d):
    """Same ordering step as the baseline modelling notebooks (Model A cell 46, Model B cell 47)."""
    return d.sort_values("subject_id").reset_index(drop=True)

data = {}
for key, d in raw.items():
    o = ordered(d)
    codes, _ = pd.factorize(o["subject_id"])          # integer patient codes; identifiers are not kept or shown
    data[key] = dict(y=o["outcome"].to_numpy(), pat=codes, n_pat=int(codes.max()) + 1)

def holdout_idx(y, seed=42):
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, test_size=0.2, random_state=seed, stratify=y)
    return tr, te

rows = []
for key, sp in SPECS.items():
    y = data[key]["y"]
    tr, te = holdout_idx(y)
    got_tr, got_te = (len(tr), int(y[tr].sum())), (len(te), int(y[te].sum()))
    rows.append({"model": key, "train (n, events) replicated": got_tr, "train expected": sp["expected_train"],
                 "test (n, events) replicated": got_te, "test expected": sp["expected_test"],
                 "match": got_tr == sp["expected_train"] and got_te == sp["expected_test"]})
table(pd.DataFrame(rows), title="Replication of the first-version hold-out split")

modelo,"entrenamiento (n, eventos) replicado",entrenamiento esperado,"prueba (n, eventos) replicado",prueba esperado,coincide
A,"(20588, 33)","(20588, 33)","(5148, 8)","(5148, 8)",Sí
B,"(20588, 380)","(20588, 380)","(5148, 95)","(5148, 95)",Sí


## E5 · Unidad de análisis y solapamiento — admisiones por paciente

In [4]:
rows = []
for key, sp in SPECS.items():
    pat, y, P = data[key]["pat"], data[key]["y"], data[key]["n_pat"]
    per_pat = np.bincount(pat, minlength=P)
    ev_pat = np.unique(pat[y == 1])
    rows.append({
        "model": key,
        "admissions": len(y),
        "patients": P,
        "patients with >1 admission": int((per_pat > 1).sum()),
        "% patients with >1 admission": round(100 * (per_pat > 1).mean(), 1),
        "median admissions per patient": float(np.median(per_pat)),
        "max admissions per patient": int(per_pat.max()),
        "positive admissions": int(y.sum()),
        "event-bearing patients (unique)": int(len(ev_pat)),
        "positive admissions per event-bearing patient: max": int(np.bincount(pat[y == 1])[ev_pat].max()),
    })
table(pd.DataFrame(rows).set_index("model").T, index=True)

dist = {}
for key in SPECS:
    per_pat = np.bincount(data[key]["pat"], minlength=data[key]["n_pat"])
    dist[key] = pd.Series(np.minimum(per_pat, 5)).value_counts().sort_index()
dist = pd.DataFrame(dist).fillna(0).astype(int)
dist.index = [f"{i} admission(s)" if i < 5 else "5 or more" for i in dist.index]
table(dist, index=True)

modelo,A,B
admisiones,"25,736","25,736"
pacientes,"15,084","15,084"
pacientes con >1 admisión,"6,793","6,793"
% de pacientes con >1 admisión,45,45
mediana de admisiones por paciente,1,1
máximo de admisiones por paciente,27,27
admisiones positivas,41,475
pacientes con evento (únicas),26,426
admisiones positivas por paciente con evento: máximo,6,7


,A,B
1 admisión(es),"8,291","8,291"
2 admisión(es),"4,418","4,418"
3 admisión(es),"1,558","1,558"
4 admisión(es),491,491
5 o más,326,326


## E5 · P2 — estadísticas de solapamiento de una partición

Para un conjunto de entrenamiento y uno de evaluación: pacientes de evaluación que también están en entrenamiento; admisiones de evaluación cuya paciente está en entrenamiento; admisiones de evaluación positivas cuya paciente tiene alguna admisión o una admisión *positiva* en entrenamiento; y pacientes con evento a ambos lados.

In [5]:
def overlap_stats(pat, y, P, tr, te):
    in_tr = np.zeros(P, dtype=bool); in_tr[pat[tr]] = True
    in_te = np.zeros(P, dtype=bool); in_te[pat[te]] = True
    pos_tr = np.zeros(P, dtype=bool); pos_tr[pat[tr][y[tr] == 1]] = True
    pos_te = np.zeros(P, dtype=bool); pos_te[pat[te][y[te] == 1]] = True
    te_pos_mask = y[te] == 1
    te_pat = pat[te]
    return {
        "train admissions": len(tr), "test admissions": len(te),
        "train events": int(y[tr].sum()), "test events": int(y[te].sum()),
        "train patients": int(in_tr.sum()), "test patients": int(in_te.sum()),
        "test patients also in train": int((in_te & in_tr).sum()),
        "% test patients also in train": 100 * (in_te & in_tr).sum() / max(in_te.sum(), 1),
        "test admissions with patient in train": int(in_tr[te_pat].sum()),
        "% test admissions with patient in train": 100 * in_tr[te_pat].mean(),
        "positive test admissions whose patient is in train": int(in_tr[te_pat[te_pos_mask]].sum()),
        "positive test admissions whose patient has a positive admission in train": int(pos_tr[te_pat[te_pos_mask]].sum()),
        "event-bearing patients in train": int(pos_tr.sum()),
        "event-bearing patients in test": int(pos_te.sum()),
        "event-bearing patients in both": int((pos_tr & pos_te).sum()),
    }

## E5 · P2 — particiones por admisión (replicadas) y simulación independiente del orden

**6.1** Reserva 80/20 (semilla 42). **6.2** Validación cruzada estratificada de cinco pliegues (con mezcla, semilla 42), cada pliegue como evaluación y los otros cuatro como entrenamiento. **6.3** Simulación: 1.000 particiones de reserva (semillas 0–999) y 200 repeticiones de validación cruzada de 5 pliegues; mediana y percentiles 2,5–97,5.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Hay pacientes a ambos lados | Todas las particiones de los notebooks 03–10 se agrupan por paciente (`StratifiedGroupKFold` por `subject_id`) | Cerca del 59% de las pacientes de prueba también estaban en entrenamiento |

In [6]:
# 6.1 hold-out, replicated
ho = {}
for key in SPECS:
    d = data[key]
    tr, te = holdout_idx(d["y"], seed=42)
    ho[key] = overlap_stats(d["pat"], d["y"], d["n_pat"], tr, te)
tab_ho = pd.DataFrame(ho)
tab_ho.columns = [f"Model {c}" for c in tab_ho.columns]
table(tab_ho.round(1), index=True, title="6.1 Hold-out 80/20 (replicated admission-level split)")

,Modelo A,Modelo B
admisiones de entrenamiento,"20,588","20,588"
admisiones de prueba,"5,148","5,148"
eventos de entrenamiento,33,380
eventos de prueba,8,95
pacientes de entrenamiento,"13,243","13,196"
pacientes de prueba,"4,527","4,523"
pacientes de prueba también en entrenamiento,"2,686","2,635"
% de pacientes de prueba también en entrenamiento,59.3,58.3
admisiones de prueba con la paciente en entrenamiento,"3,086","3,065"
% de admisiones de prueba con la paciente en entrenamiento,59.9,59.5


In [7]:
# 6.2 five-fold CV, replicated (all folds shown, then the sum over folds)
cv_tabs = {}
for key in SPECS:
    d = data[key]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    X_dummy = np.zeros(len(d["y"]))
    fold_rows = []
    for k, (tr, te) in enumerate(skf.split(X_dummy, d["y"]), 1):
        s = overlap_stats(d["pat"], d["y"], d["n_pat"], tr, te)
        s["fold"] = k
        fold_rows.append(s)
    cv_tabs[key] = pd.DataFrame(fold_rows).set_index("fold")
    table(cv_tabs[key][["test admissions", "test events", "test patients", "test patients also in train",
                          "% test patients also in train", "% test admissions with patient in train",
                          "positive test admissions whose patient is in train",
                          "positive test admissions whose patient has a positive admission in train",
                          "event-bearing patients in both"]].round(1), index=True,
          title=f"6.2 Model {key}: 5-fold CV, per fold")

,admisiones de prueba,eventos de prueba,pacientes de prueba,pacientes de prueba también en entrenamiento,% de pacientes de prueba también en entrenamiento,% de admisiones de prueba con la paciente en entrenamiento,admisiones de prueba positivas cuya paciente está en entrenamiento,admisiones de prueba positivas cuya paciente tiene una admisión positiva en entrenamiento,pacientes con evento en ambos lados
pliegue,,,,,,,,,
1,"5,148",9,"4,509","2,670",59.2,59.9,6,6,4
2,"5,147",8,"4,555","2,739",60.1,60.7,5,5,4
3,"5,147",8,"4,558","2,699",59.2,60.2,4,4,4
4,"5,147",8,"4,537","2,680",59.1,60,5,5,5
5,"5,147",8,"4,525","2,649",58.5,59.4,3,3,3


,admisiones de prueba,eventos de prueba,pacientes de prueba,pacientes de prueba también en entrenamiento,% de pacientes de prueba también en entrenamiento,% de admisiones de prueba con la paciente en entrenamiento,admisiones de prueba positivas cuya paciente está en entrenamiento,admisiones de prueba positivas cuya paciente tiene una admisión positiva en entrenamiento,pacientes con evento en ambos lados
pliegue,,,,,,,,,
1,"5,148",95,"4,553","2,683",58.9,59.6,47,17,14
2,"5,147",95,"4,520","2,723",60.2,61.3,49,12,12
3,"5,147",95,"4,542","2,649",58.3,59,50,18,17
4,"5,147",95,"4,539","2,746",60.5,61.9,53,10,9
5,"5,147",95,"4,562","2,694",59.1,59.7,45,10,8


In [8]:
# 6.3 simulation (order-independent)
def summarize(rows):
    df = pd.DataFrame(rows)
    out = {}
    for c in df.columns:
        lo, med, hi = np.percentile(df[c], [2.5, 50, 97.5])
        out[c] = f"{med:.1f} ({lo:.1f}-{hi:.1f})"
    return pd.Series(out)

KEYS = ["% test patients also in train", "% test admissions with patient in train",
        "positive test admissions whose patient is in train",
        "positive test admissions whose patient has a positive admission in train",
        "event-bearing patients in both"]

sim_ho, sim_cv = {}, {}
for key in SPECS:
    d = data[key]
    rows = []
    for seed in range(1000):
        tr, te = holdout_idx(d["y"], seed=seed)
        s = overlap_stats(d["pat"], d["y"], d["n_pat"], tr, te)
        rows.append({k: s[k] for k in KEYS})
    sim_ho[f"Model {key}"] = summarize(rows)

    rows = []
    for rep in range(200):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=rep)
        agg = {k: [] for k in KEYS}
        for tr, te in skf.split(np.zeros(len(d["y"])), d["y"]):
            s = overlap_stats(d["pat"], d["y"], d["n_pat"], tr, te)
            for k in KEYS:
                agg[k].append(s[k])
        # per repetition: mean of the percentages, sum of the counts over the 5 folds
        rows.append({k: (np.mean(v) if k.startswith("%") else np.sum(v)) for k, v in agg.items()})
    sim_cv[f"Model {key}"] = summarize(rows)

table(pd.DataFrame(sim_ho), index=True,
      title="6.3a Hold-out: 1000 random stratified 80/20 splits -- median (2.5th-97.5th percentile)")
table(pd.DataFrame(sim_cv), index=True,
      title="6.3b 5-fold CV: 200 repetitions -- median (2.5th-97.5th percentile); % = mean over folds, counts = summed over folds")

,Modelo A,Modelo B
% de pacientes de prueba también en entrenamiento,59.2 (58.0-60.4),59.2 (58.1-60.5)
% de admisiones de prueba con la paciente en entrenamiento,60.1 (58.9-61.3),60.1 (58.8-61.4)
admisiones de prueba positivas cuya paciente está en entrenamiento,4.0 (2.0-7.0),48.0 (39.0-57.0)
admisiones de prueba positivas cuya paciente tiene una admisión positiva en entrenamiento,4.0 (2.0-7.0),14.0 (8.0-20.0)
pacientes con evento en ambos lados,4.0 (1.0-6.0),12.0 (7.0-17.0)


,Modelo A,Modelo B
% de pacientes de prueba también en entrenamiento,59.2 (58.9-59.6),59.2 (58.9-59.6)
% de admisiones de prueba con la paciente en entrenamiento,60.1 (59.7-60.6),60.1 (59.7-60.5)
admisiones de prueba positivas cuya paciente está en entrenamiento,21.0 (18.0-23.0),240.0 (230.0-249.0)
admisiones de prueba positivas cuya paciente tiene una admisión positiva en entrenamiento,21.0 (18.0-23.0),72.0 (63.0-78.0)
pacientes con evento en ambos lados,18.0 (15.0-21.0),61.0 (53.0-68.1)


## E5 · E11 Reporte — resumen compacto

Un bloque corto con las cifras de solapamiento. Solo agregados.

In [9]:
summary = {}
for key, sp in SPECS.items():
    d = data[key]; P = d["n_pat"]
    per_pat = np.bincount(d["pat"], minlength=P)
    s = ho[key]
    cvt = cv_tabs[key]
    match = all([
        (s["train admissions"], s["train events"]) == sp["expected_train"],
        (s["test admissions"], s["test events"]) == sp["expected_test"],
    ])
    sh = sim_ho["Model " + key]
    summary[f"Model {key}"] = {
        "admissions": len(d["y"]),
        "patients": P,
        "patients with >1 admission (%)": f"{int((per_pat > 1).sum())} ({100 * (per_pat > 1).mean():.1f}%)",
        "positive admissions": int(d["y"].sum()),
        "event-bearing patients (unique)": len(np.unique(d["pat"][d["y"] == 1])),
        "hold-out replication matches expected sizes and events": match,
        "hold-out: test patients also in train": f"{s['test patients also in train']}/{s['test patients']} "
                                                 f"({s['% test patients also in train']:.1f}%)",
        "hold-out: test admissions with patient in train": f"{s['test admissions with patient in train']}/{s['test admissions']} "
                                                           f"({s['% test admissions with patient in train']:.1f}%)",
        "hold-out: positive test admissions whose patient is in train":
            f"{s['positive test admissions whose patient is in train']}/{s['test events']}",
        "hold-out: positive test admissions whose patient has a positive admission in train":
            s["positive test admissions whose patient has a positive admission in train"],
        "hold-out: event-bearing patients train/test/both":
            f"{s['event-bearing patients in train']}/{s['event-bearing patients in test']}/{s['event-bearing patients in both']}",
        "5-fold CV: mean % test patients also in train": round(cvt["% test patients also in train"].mean(), 1),
        "5-fold CV: mean % test admissions with patient in train": round(cvt["% test admissions with patient in train"].mean(), 1),
        "5-fold CV: positive test admissions whose patient is in train (sum of folds)":
            f"{int(cvt['positive test admissions whose patient is in train'].sum())}/{int(cvt['test events'].sum())}",
        "5-fold CV: positive test admissions whose patient has a positive admission in train (sum of folds)":
            int(cvt["positive test admissions whose patient has a positive admission in train"].sum()),
        "simulation, 1000 hold-out splits: % test admissions with patient in train": sh["% test admissions with patient in train"],
        "simulation, 1000 hold-out splits: positive test admissions whose patient is in train":
            sh["positive test admissions whose patient is in train"],
    }
table(pd.DataFrame(summary), index=True, title="Compact summary of patient overlap (aggregates only)")

,Modelo A,Modelo B
admisiones,"25,736","25,736"
pacientes,"15,084","15,084"
pacientes con >1 admisión (%),6793 (45.0%),6793 (45.0%)
admisiones positivas,41,475
pacientes con evento (únicas),26,426
la réplica de la reserva coincide en tamaños y eventos,Sí,Sí
reserva: pacientes de prueba también en entrenamiento,2686/4527 (59.3%),2635/4523 (58.3%)
reserva: admisiones de prueba con la paciente en entrenamiento,3086/5148 (59.9%),3065/5148 (59.5%)
reserva: admisiones positivas de prueba con la paciente en entrenamiento,6/8,48/95
reserva: admisiones positivas de prueba con la paciente con una admisión positiva en entrenamiento,6,13


## Resumen del notebook

La puerta P2 con las particiones por admisión de la primera versión.

In [10]:
# Summary of notebook 02
notebook_summary([
    ("Model B (SMM): test patients also in train, hold-out", lambda: f"{ho['B']['% test patients also in train']:.1f}%"),
    ("Model B (SMM): positive test admissions whose patient is in train, hold-out",
     lambda: f"{ho['B']['positive test admissions whose patient is in train']}/{ho['B']['test events']}"),
    ("Model A (death): positive test admissions whose patient is in train, hold-out",
     lambda: f"{ho['A']['positive test admissions whose patient is in train']}/{ho['A']['test events']}"),
    ("gate G2 with admission-level splits", lambda: "does NOT pass: the same patient is on both sides"
     if ho["B"]["test patients also in train"] > 0 else "passes"),
    ("decision", "every split of notebooks 03 to 10 is grouped by patient (StratifiedGroupKFold)"),
], title="Notebook summary: gate G2")

,valor
"Modelo B (SMM): pacientes de prueba también en entrenamiento, reserva",58.3%
"Modelo B (SMM): admisiones positivas de prueba con la paciente en entrenamiento, reserva",48/95
"Modelo A (muerte): admisiones positivas de prueba con la paciente en entrenamiento, reserva",6/8
puerta P2 con particiones por admisión,NO pasa: la misma paciente queda a ambos lados
decisión,todas las particiones de los notebooks 03 a 10 se agrupan por paciente (StratifiedGroupKFold)
